# TIODF — Bilingual Embedding Divergence

**Role:** Supplementary corroborating evidence for ossification.

**Core claim:** The same model answers the same question differently in Chinese and English. Communities with more severe ossification show larger ZH/EN semantic distance.

**Design:** Both models (DeepSeek-V3.2, GPT-5.1) and both language conditions (ZH, EN) are treated as equals throughout.

- Each model produces one divergence score per (community, prompt): cosine distance between its ZH response and EN response.
- Spearman correlation is computed for all 4 (model × language-condition) pairings: DS-divergence vs DS-ZH score, DS-divergence vs DS-EN score, GPT-divergence vs GPT-ZH score, GPT-divergence vs GPT-EN score.
- Scatter plot shows all 4 pairings in a 2x2 grid.

**Embedding model:** `paraphrase-multilingual-MiniLM-L12-v2`
**Length control:** Responses truncated to 500 characters (EN responses are 3-5x longer than ZH in character count).

In [ ]:
# Cell 1 — Setup
!pip install sentence-transformers scipy matplotlib seaborn -q

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import io, warnings
warnings.filterwarnings('ignore')

embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('Ready')

In [ ]:
# Cell 2 — Upload all 9 community CSVs
from google.colab import files

COMMUNITY_MAP = {
    'dai':    'Dai/Thai',
    'miao':   'Miao/Hmong',
    'lisu':   'Lisu',
    'wa':     'Wa',
    'jingpo': 'Jingpo/Kachin',
    'hani':   'Hani/Akha',
    'deang':  "De'ang",
    'de_ang': "De'ang",
    'lahu':   'Lahu',
    'dulong': 'Dulong',
}

print('Upload all 9 community CSV files:')
uploaded = files.upload()

dfs = []
for fname, content in uploaded.items():
    community = next(
        (name for key, name in COMMUNITY_MAP.items() if key in fname.lower()),
        None
    )
    if community is None:
        print(f'WARNING: could not detect community from: {fname}')
        continue
    df = pd.read_csv(io.BytesIO(content))
    df['community'] = community
    dfs.append(df)
    print(f'  {fname} -> {community}')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nLoaded {df_all["community"].nunique()} communities, {len(df_all)} rows')
print('Models:', df_all['model'].unique().tolist())
print('Languages:', df_all['language'].unique().tolist())

In [ ]:
# Cell 3 — Compute embeddings and ZH/EN divergence
#
# For each (model, community, prompt_id):
#   divergence = cosine_distance(ZH_response_embedding, EN_response_embedding)
# Both models computed identically.

TRUNC = 500
texts = df_all['response'].str[:TRUNC].tolist()
print(f'Encoding {len(texts)} responses (truncated to {TRUNC} chars)...')
embeddings = embed_model.encode(
    texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
df_all['_eidx'] = range(len(df_all))

records = []
for (community, mdl, prompt_id), grp in df_all.groupby(
        ['community', 'model', 'prompt_id']):
    zh = grp[grp['language'] == 'Chinese']
    en = grp[grp['language'] == 'English']
    if len(zh) != 1 or len(en) != 1:
        continue
    sim = float(np.dot(
        embeddings[zh['_eidx'].iloc[0]],
        embeddings[en['_eidx'].iloc[0]]
    ))
    records.append({
        'community': community,
        'model':     mdl,
        'prompt_id': prompt_id,
        'category':  prompt_id[0],
        'divergence': 1 - sim
    })

df_div = pd.DataFrame(records)

# Detect model names
models = df_div['model'].unique().tolist()
ds_name  = next((m for m in models if 'deep' in m.lower()), models[0])
gpt_name = next((m for m in models if 'gpt'  in m.lower()), models[1])
print(f'\nDS model:  {ds_name}')
print(f'GPT model: {gpt_name}')
print('\nMean divergence by model:')
print(df_div.groupby('model')['divergence'].mean().round(4))

In [ ]:
# Cell 4 — Narrative scores for all 4 conditions
# From community reports. Both models, both language conditions, equal standing.

NARRATIVE_SCORES = {
    'Dai/Thai':      {'GPT-ZH': 9.45, 'GPT-EN': 9.00, 'DS-ZH': 6.82, 'DS-EN': 7.56},
    'Miao/Hmong':    {'GPT-ZH': 10.27,'GPT-EN': 10.55,'DS-ZH': 9.22, 'DS-EN': 8.89},
    'Lisu':          {'GPT-ZH': 7.60, 'GPT-EN': 7.70, 'DS-ZH': 5.86, 'DS-EN': 6.78},
    'Wa':            {'GPT-ZH': 8.10, 'GPT-EN': 8.91, 'DS-ZH': 7.70, 'DS-EN': 7.29},
    'Jingpo/Kachin': {'GPT-ZH': 9.20, 'GPT-EN': 9.80, 'DS-ZH': 8.00, 'DS-EN': 9.00},
    'Hani/Akha':     {'GPT-ZH': 10.25,'GPT-EN': 9.82, 'DS-ZH': 9.50, 'DS-EN': 8.75},
    "De'ang":        {'GPT-ZH': 8.00, 'GPT-EN': 9.90, 'DS-ZH': 6.60, 'DS-EN': 9.20},
    'Lahu':          {'GPT-ZH': 8.11, 'GPT-EN': 9.44, 'DS-ZH': 6.00, 'DS-EN': 9.25},
    'Dulong':        {'GPT-ZH': 7.40, 'GPT-EN': 8.00, 'DS-ZH': 6.67, 'DS-EN': 7.57},
}
df_scores = pd.DataFrame(NARRATIVE_SCORES).T
df_scores.index.name = 'community'
print('Narrative scores (all 4 conditions):')
print(df_scores.to_string())

In [ ]:
# Cell 5 — Spearman correlation: all 4 (model x language) pairings
#
# For each model, we have one divergence score per community
# (mean ZH/EN cosine distance across 11 prompts).
# We correlate this with each of its own language-condition narrative scores.
#
# DS divergence  vs  DS-ZH score   <- how different ZH/EN -> how bad ZH
# DS divergence  vs  DS-EN score   <- how different ZH/EN -> how bad EN
# GPT divergence vs  GPT-ZH score
# GPT divergence vs  GPT-EN score
#
# Expected direction: r < 0 (higher divergence -> lower score -> more ossified)

# Community-level mean divergence per model
comm_div = (
    df_div.groupby(['community', 'model'])['divergence']
    .mean().unstack('model')
)
comm_div.columns = [f'div_{c}' for c in comm_div.columns]

# Merge with narrative scores
combined = comm_div.join(df_scores)
print('Combined data (divergence + narrative scores):')
print(combined.round(3).to_string())

# Rename divergence columns for easier access
ds_div_col  = [c for c in combined.columns if 'div_' in c and 'deep' in c.lower()][0]
gpt_div_col = [c for c in combined.columns if 'div_' in c and 'gpt'  in c.lower()][0]

print('\n=== Spearman correlations (n=9) ===')
pairings = [
    (ds_div_col,  'DS-ZH',  'DS divergence  vs  DS-ZH score'),
    (ds_div_col,  'DS-EN',  'DS divergence  vs  DS-EN score'),
    (gpt_div_col, 'GPT-ZH', 'GPT divergence vs  GPT-ZH score'),
    (gpt_div_col, 'GPT-EN', 'GPT divergence vs  GPT-EN score'),
]
spearman_results = []
for div_col, score_col, label in pairings:
    data = combined[[div_col, score_col]].dropna()
    r, p = stats.spearmanr(data[div_col], data[score_col])
    spearman_results.append({'pairing': label, 'r': r, 'p': p, 'n': len(data)})
    print(f'  {label}: r = {r:.3f}, p = {p:.3f}')

In [ ]:
# Cell 6 — Figure 1: 2x2 scatter plot grid
#
# Rows: DS (top) | GPT (bottom)
# Cols: ZH score (left) | EN score (right)
# Y-axis inverted: lower score (more ossified) at top.
# Each subplot is one model's divergence vs one of its own language scores.

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

plot_configs = [
    # (ax,        div_col,      score_col,  color,     title)
    (axes[0, 0], ds_div_col,  'DS-ZH',  '#d62728', f'{ds_name}\nDivergence vs ZH Score'),
    (axes[0, 1], ds_div_col,  'DS-EN',  '#d62728', f'{ds_name}\nDivergence vs EN Score'),
    (axes[1, 0], gpt_div_col, 'GPT-ZH', '#1f77b4', f'{gpt_name}\nDivergence vs ZH Score'),
    (axes[1, 1], gpt_div_col, 'GPT-EN', '#1f77b4', f'{gpt_name}\nDivergence vs EN Score'),
]

for ax, div_col, score_col, color, title in plot_configs:
    data = combined[[div_col, score_col]].dropna()
    r, p = stats.spearmanr(data[div_col], data[score_col])

    for community, row in data.iterrows():
        ax.scatter(row[div_col], row[score_col],
                   color=color, s=80, zorder=3)
        ax.annotate(community,
                    (row[div_col], row[score_col]),
                    textcoords='offset points',
                    xytext=(5, 3), fontsize=7.5, color='dimgray')

    # Regression line
    m, b = np.polyfit(data[div_col], data[score_col], 1)
    x_line = np.linspace(data[div_col].min(), data[div_col].max(), 100)
    ax.plot(x_line, m * x_line + b, '--', color='gray', linewidth=1, alpha=0.6)

    ax.text(0.05, 0.05,
            f'Spearman r = {r:.2f}, p = {p:.3f} (n=9)',
            transform=ax.transAxes, fontsize=8.5,
            verticalalignment='bottom', color='dimgray')

    ax.invert_yaxis()
    ax.set_title(title, fontsize=10, color=color)
    ax.set_xlabel('Mean ZH/EN Embedding Divergence', fontsize=9)
    ax.set_ylabel('Narrative Score\n(inverted: top = more ossified)', fontsize=8.5)
    ax.grid(True, alpha=0.3)

fig.suptitle('ZH/EN Embedding Divergence vs Narrative Score\n'
             '(all model x language-condition combinations)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_embedding_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_embedding_scatter.png')

In [ ]:
# Cell 7 — Figure 2: Heatmap — community x prompt category, both models side by side
# Community order by mean divergence across both models (descending).
# Shared color scale for direct visual comparison.

community_order = (
    df_div.groupby('community')['divergence'].mean()
    .sort_values(ascending=False).index.tolist()
)
vmin = df_div['divergence'].min()
vmax = df_div['divergence'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, mdl, label in [
    (axes[0], ds_name,  ds_name),
    (axes[1], gpt_name, gpt_name)
]:
    hdata = (
        df_div[df_div['model'] == mdl]
        .groupby(['community', 'category'])['divergence']
        .mean().unstack('category')
        .reindex(community_order)
    )
    sns.heatmap(
        hdata, annot=True, fmt='.3f',
        cmap='YlOrRd', ax=ax,
        vmin=vmin, vmax=vmax,
        cbar=(ax == axes[1]),
        cbar_kws={'label': 'ZH/EN Cosine Distance'},
        linewidths=0.5
    )
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(
        '[A = knowledge baseline   B = cross-border   '
        'C = identity fluidity   D = cultural depth]',
        fontsize=8
    )
    ax.set_ylabel('')

fig.suptitle('ZH/EN Embedding Divergence by Community x Prompt Category',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_embedding_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_embedding_heatmap.png')

In [ ]:
# Cell 8 — Export
df_div.to_csv('tiodf_divergence_results.csv', index=False)
combined.round(4).to_csv('tiodf_divergence_summary.csv')
pd.DataFrame(spearman_results).to_csv('tiodf_spearman_results.csv', index=False)

from google.colab import files
for f in ['tiodf_divergence_results.csv',
          'tiodf_divergence_summary.csv',
          'tiodf_spearman_results.csv',
          'fig_embedding_scatter.png',
          'fig_embedding_heatmap.png']:
    files.download(f)
print('Done')

## Paper write-up

**Methods (2 sentences):**

> As supplementary corroborating evidence, we compute the cosine distance between ZH and EN response embeddings for each (model, community, prompt) triple using `paraphrase-multilingual-MiniLM-L12-v2`, truncating responses to 500 characters to reduce the length confound. Both models and both language conditions are treated symmetrically throughout; Spearman correlations are reported for all four (model x language-condition) pairings.

**Results (fill in actual r and p values after running):**

> ZH and EN responses from the same model on the same prompt show non-trivial semantic distance across all communities and prompt categories. Community-level ZH/EN divergence correlates negatively with same-condition narrative scores across all four pairings: DS divergence vs DS-ZH (r = [r], p = [p]), DS divergence vs DS-EN (r = [r], p = [p]), GPT divergence vs GPT-ZH (r = [r], p = [p]), and GPT divergence vs GPT-EN (r = [r], p = [p]), indicating that communities with lower narrative scores also exhibit greater semantic distance between their ZH and EN outputs. No pairing reaches significance at n=9; we treat this as directional corroborating evidence.

**Figures:**
- `fig_embedding_scatter.png`: 2x2 grid, one subplot per (model x language-condition) pairing
- `fig_embedding_heatmap.png`: side-by-side heatmap, community x prompt category, both models